In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch_geometric.data import Data as PyGGraph

In [3]:
edges_from = [2,1,0,4]
edges_to = [1,2,4,0]
edge_index = torch.tensor([edges_from, edges_to], dtype=torch.long)
positions = torch.tensor([[0, 0], [1, 0], [0, 1], [1, 1], [2, 0]], dtype=torch.float)
g= PyGGraph(edge_index=edge_index, pos=positions)
g

Data(edge_index=[2, 4], pos=[5, 2])

In [4]:
temperatures = torch.tensor([300, 400, 500, 600, 700], dtype=torch.float)
g["tempr"] = temperatures
g

Data(edge_index=[2, 4], pos=[5, 2], tempr=[5])

In [5]:
distances = torch.norm(g.pos[edges_from] - g.pos[edges_to], dim=1)
distances

tensor([1.4142, 1.4142, 2.0000, 2.0000])

In [6]:
g.edge_attr = distances.unsqueeze(1)

In [7]:
g.node_attrs()

['tempr', 'pos']

In [8]:
import cooldata.utils as cdu
from cooldata.pyvista_flow_field_dataset import PyvistaFlowFieldDataset
ds_pv = PyvistaFlowFieldDataset.load_from_huggingface(
    num_samples=12, data_dir="datasets/pyvista"
)
sd =ds_pv[0].surface_data

Loaded 12 samples from '/Users/ole/Documents/software/flow_field_dataset/examples/dataset_exploration/datasets/pyvista'.
Loaded 12 samples from 'datasets/pyvista'.


In [59]:
import pyransame

combined = sd[0].combine()
random_points =pyransame.random_surface_dataset(combined,n=10000)
random_points.plot(scalars="Pressure")

Widget(value='<iframe src="http://localhost:57801/index.html?ui=P_0x38ab83bd0_19&reconnect=auto" class="pyvist…

In [61]:
surface = random_points.reconstruct_surface()
surface.plot()

Widget(value='<iframe src="http://localhost:57801/index.html?ui=P_0x38aae27d0_20&reconnect=auto" class="pyvist…

In [46]:
import pyvista as pv
import numpy as np

def sample_and_connect_surface_points(mesh: pv.PolyData, n=100, k=3):
    rng = np.random.default_rng()
    tri = mesh.extract_surface().triangulate()

    # Use largest connected component only:
    tri = tri.connectivity(largest=True)

    areas = tri.compute_cell_sizes()['Area']
    cdf = np.cumsum(areas)
    picks = np.searchsorted(cdf, rng.random(n) * cdf[-1])
    con = tri.faces.reshape(-1, 4)[:, 1:]
    corners = tri.points[con[picks]]
    u = rng.random(n)
    v = rng.random(n)
    w = 1.0 - np.sqrt(u)
    s = np.sqrt(u) * (1.0 - v)
    t = np.sqrt(u) * v
    sampled_pts = corners[:, 0] * w[:, None] + corners[:, 1] * s[:, None] + corners[:, 2] * t[:, None]

    closest_indices = np.array([tri.find_closest_point(pt) for pt in sampled_pts])
    labels = tri.point_data['RegionId']

    lines = []
    for i in range(n):
        dists = np.full(n, np.inf)
        for j in range(n):
            if i == j:
                continue
            if labels[closest_indices[i]] != labels[closest_indices[j]]:
                continue  # skip different components

            try:
                path = tri.geodesic(closest_indices[i], closest_indices[j])
                dists[j] = path.length
            except RuntimeError:
                pass

        neighbors = np.argsort(dists)[:k]
        for j in neighbors:
            if dists[j] != np.inf:
                lines.append([2, i, j])

    sampled_poly = pv.PolyData(sampled_pts)
    graph = pv.PolyData(sampled_pts, lines=np.array(lines))
    return sampled_poly, graph
poly, graph =sample_and_connect_surface_points(combined, n=50, k=4)

/Users/ole/Documents/software/flow_field_dataset/.venv/lib/python3.11/site-packages/pyvista/core/filters/data_set.py:1963: PyVistaDeprecationWarning: Use of `largest=True` is deprecated. Use 'largest' or `extraction_mode='largest'` instead.
  warnings.warn(


In [49]:
poly.plot()

Widget(value='<iframe src="http://localhost:57801/index.html?ui=P_0x3123a2350_12&reconnect=auto" class="pyvist…

In [ ]:
mesh=combined
tri=mesh.extract_surface().triangulate()
tri.cell

TypeError: 'generator' object is not callable